# Requirement
    *Python version 3.9.20*
    *USB 3.1*
    *pyrealsense library*

# Camera Check

In [ ]:
# Cell 0 — Import & environment
import sys, platform
print(f"Python: {sys.version.split()[0]}  |  Platform: {platform.system()} {platform.release()}")

import pyrealsense2 as rs
print("pyrealsense2 import: OK")

# Optional: confirm the wheel version via pkg_resources (works when installed via pip)
try:
    import pkg_resources
    print("pyrealsense2 package version:", pkg_resources.get_distribution("pyrealsense2").version)
except Exception:
    pass


In [ ]:
# Cell 1 — List devices
import pyrealsense2 as rs

ctx = rs.context()
devices = ctx.query_devices()

if len(devices) == 0:
    raise RuntimeError("No RealSense devices found. Check USB 3.x port/cable, try replugging, and avoid unpowered hubs.")

print(f"Found {len(devices)} RealSense device(s).")
l500 = None
for i, dev in enumerate(devices):
    name = dev.get_info(rs.camera_info.name) if dev.supports(rs.camera_info.name) else "Unknown"
    serial = dev.get_info(rs.camera_info.serial_number) if dev.supports(rs.camera_info.serial_number) else "Unknown"
    product_line = dev.get_info(rs.camera_info.product_line) if dev.supports(rs.camera_info.product_line) else "Unknown"
    fw = dev.get_info(rs.camera_info.firmware_version) if dev.supports(rs.camera_info.firmware_version) else "Unknown"
    usb = dev.get_info(rs.camera_info.usb_type_descriptor) if dev.supports(rs.camera_info.usb_type_descriptor) else "Unknown"
    print(f"[{i}] {name} | S/N: {serial} | Line: {product_line} | FW: {fw} | USB: {usb}")
    if product_line == "L500":
        l500 = dev

if l500 is None:
    raise RuntimeError("No L500-series device found. Ensure the L515 is connected and recognized by the OS.")
else:
    print("L500 device detected ✅")


**Camera Configuration**

In [ ]:
import pyrealsense2 as rs

# Initialize pipeline and start configuration
pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)
config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)

# Start the pipeline
pipeline.start(config)

# Get the depth sensor
device = pipeline.get_active_profile().get_device()
depth_sensor = device.first_depth_sensor()

# List supported options
print("Supported options and current values:")
for option_name in dir(rs.option):
    if not option_name.startswith('__'):  # Skip private/internal attributes
        try:
            option = getattr(rs.option, option_name)  # Get option by name
            if depth_sensor.supports(option):
                current_value = depth_sensor.get_option(option)
                print(f"{option_name}: {current_value}")
        except Exception as e:
            print(f"Could not query option {option_name}: {e}")

# Stop the pipeline
pipeline.stop()

**Configuration Support**

In [ ]:
def check_option_support(sensor, option_name):
    """Check if a specific option is supported by the sensor."""
    option = getattr(rs.option, option_name, None)
    if option and sensor.supports(option):
        print(f"{option_name} is supported.")
        option_range = sensor.get_option_range(option)
        print(f"Range: Min = {option_range.min}, Max = {option_range.max}, Default = {option_range.default}")
    else:
        print(f"{option_name} is not supported.")

# Check laser_power and exposure
check_option_support(depth_sensor, "laser_power")
check_option_support(depth_sensor, "exposure")

# Center-Distance Calibration

*Center-Distance Log Validation*

In [ ]:
import pyrealsense2 as rs
import numpy as np
import cv2 as cv
import time
import matplotlib.pyplot as plt
from datetime import datetime
import glob
import csv

# Configuration Parameters
DEPTH_MIN, DEPTH_MAX = 0.49, 3.5               # Depth range (m)
DEPTH_FOV_H, DEPTH_FOV_V = 70, 55              # Degrees
WIDTH, HEIGHT = 640, 480                       # Frame resolution
FPS = 30
ROI_WIDTH_M, ROI_HEIGHT_M = 0.02, 0.02         # ROI size in meters
SPATIAL_THRESHOLD = 0.20                      # Dynamic depth margin ratio
DURATION_S = 60                                # Total recording duration (seconds)
INTERVAL_S = 1                                 # Recording interval (seconds)

# State variables
depth_scale = None
pipeline = None

def init_camera():
    global pipeline, depth_scale
    pipeline = rs.pipeline()
    config = rs.config()
    config.enable_stream(rs.stream.depth, WIDTH, HEIGHT, rs.format.z16, FPS)
    config.enable_stream(rs.stream.color, WIDTH, HEIGHT, rs.format.bgr8, FPS)
    profile = pipeline.start(config)
    sensor = profile.get_device().first_depth_sensor()
    depth_scale = sensor.get_depth_scale()

# Main
if __name__ == '__main__':
    init_camera()
    cx, cy = WIDTH // 2, HEIGHT // 2

    ref_dist = None
    LOG_FILENAME = None
    records = []
    recording = False
    start_time = None
    next_record_time = None
    iteration = 0
    max_iterations = int(DURATION_S / INTERVAL_S)

    print("Press 'f' to set reference and filename, 's' to start recording, 'q' to quit.")
    cv.namedWindow("ROI Measurement", cv.WINDOW_AUTOSIZE)

    try:
        while True:
            current_time = time.time()
            # fetch frames
            frames = pipeline.wait_for_frames()
            depth_frame = frames.get_depth_frame()
            color_frame = frames.get_color_frame()
            if not depth_frame or not color_frame:
                continue

            # dynamic ROI
            d_center = depth_frame.get_distance(cx, cy) or DEPTH_MAX
            mpp_x = 2 * d_center * np.tan(np.radians(DEPTH_FOV_H/2)) / WIDTH
            mpp_y = 2 * d_center * np.tan(np.radians(DEPTH_FOV_V/2)) / HEIGHT
            rw = max(1, int(ROI_WIDTH_M / mpp_x))
            rh = max(1, int(ROI_HEIGHT_M / mpp_y))
            x1 = max(0, cx - rw//2)
            y1 = max(0, cy - rh//2)
            x2 = min(WIDTH-1, cx + rw//2)
            y2 = min(HEIGHT-1, cy + rh//2)

            # compute avg
            depth_arr = np.asanyarray(depth_frame.get_data(), dtype=np.float32) * depth_scale
            roi = depth_arr[y1:y2+1, x1:x2+1]
            zmin = max(DEPTH_MIN, d_center * (1 - SPATIAL_THRESHOLD))
            zmax = min(DEPTH_MAX, d_center * (1 + SPATIAL_THRESHOLD))
            mask = (roi >= zmin) & (roi <= zmax)
            vals = roi[mask]
            avg_mm = float(np.mean(vals)) * 1000 if vals.size else 0.0

            # display
            img = np.asanyarray(color_frame.get_data())
            cv.rectangle(img, (x1, y1), (x2, y2), (0,255,0), 2)
            cv.putText(img, f"Avg: {avg_mm:.1f} mm", (x1, y1-10), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
            if ref_dist is not None:
                diff = avg_mm - ref_dist
                cv.putText(img, f"error {diff:.1f} mm", (x1, y2+20), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)
            cv.imshow("ROI Measurement", img)

            # handle recording
            if recording and current_time >= next_record_time:
                if iteration >= max_iterations:
                    recording = False
                    print("Recording complete.")
                else:
                    # ensure ref and filename
                    if ref_dist is None:
                        print("Set reference first with 'f'.")
                        recording = False
                    else:
                        diff = avg_mm - ref_dist
                        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                        records.append((ts, avg_mm, diff))
                        with open(LOG_FILENAME, 'a') as f:
                            f.write(f"{ts},{avg_mm:.2f},{diff:.2f}\n")
                        print(f"[{iteration+1}] {ts}, Avg={avg_mm:.2f} mm, Δ={diff:.2f} mm")
                        iteration += 1
                        next_record_time += INTERVAL_S

            # key input
            key = cv.waitKey(1) & 0xFF
            if key == ord('f'):
                ref_dist = avg_mm
                ref_mm = int(round(ref_dist))
                LOG_FILENAME = f"log_{ref_mm}mm.csv"
                # write header
                with open(LOG_FILENAME, 'w') as f:
                    f.write("timestamp,measured_mm,error_mm\n")
                print(f"Reference set: {ref_dist:.1f} mm → logging to {LOG_FILENAME}")
            elif key == ord('s') and not recording:
                if LOG_FILENAME is None:
                    print("Set reference first (press 'f').")
                else:
                    recording = True
                    start_time = time.time()
                    next_record_time = start_time
                    iteration = 0
                    records.clear()
                    print("Started automatic recording...")
            elif key == ord('q'):
                break

    finally:
        pipeline.stop()
        cv.destroyAllWindows()

        # post-process if any records
        if records:
            # Load and group errors per file
            paths = sorted(glob.glob('log_*mm.csv'))
            errors_by_ref = {}
            for path in paths:
                ref = int(path.split('_')[1].rstrip('mm.csv'))
                errs = []
                with open(path) as f:
                    reader = csv.DictReader(f)
                    for row in reader:
                        errs.append(float(row['error_mm']))
                errors_by_ref[ref] = errs

            # Plot boxplots
            refs = sorted(errors_by_ref.keys())
            data = [errors_by_ref[r] for r in refs]
            plt.figure(figsize=(12,6))
            plt.boxplot(data, positions=refs, widths=3, whis=[5,95])
            plt.xlabel('Reference Distance (mm)')
            plt.ylabel('Measurement Error (mm)')
            plt.title('Error Distribution vs. Reference Distance')
            plt.grid(True, linestyle='--', alpha=0.5)
            plt.tight_layout()
            plt.show()

            # Compute and print metrics for each reference
            for ref in refs:
                errs = np.array(errors_by_ref[ref])
                bias = errs.mean()
                std = errs.std(ddof=1)
                mean_dist = ref + bias
                cov = 1.96 * std
                lower = mean_dist - cov
                upper = mean_dist + cov
                print(f"Ref={ref} mm: Bias={bias:.2f} mm, Uncertainty={std:.2f} mm, ")
                print(f"    Mean={mean_dist:.2f} mm ±{cov:.2f} mm; [{lower:.2f}, {upper:.2f}] mm")

**Centre Log CSV Only**

In [ ]:
# --- CSV-only Exploration Script with Table Output ---
def explore_csv_logs(csv_pattern='log_*mm.csv'):
    """Load all log CSVs matching pattern, produce boxplot + metrics table, and export CSV."""
    import glob, csv, numpy as np, matplotlib.pyplot as plt
    import pandas as pd

    errors_by_ref = {}
    metrics = []

    for path in sorted(glob.glob(csv_pattern)):
        # parse reference distance
        ref = int(path.split('_')[1].rstrip('mm.csv'))
        errs = []
        with open(path) as f:
            reader = csv.DictReader(f)
            for row in reader:
                errs.append(float(row['error_mm']))
        errors_by_ref[ref] = errs

        # calculate metrics
        errs_np = np.array(errs)
        bias = errs_np.mean()
        std = errs_np.std(ddof=1)
        mean_dist = ref + bias
        cov = 1.96 * std
        lower = mean_dist - cov
        upper = mean_dist + cov
        metrics.append({
            "Reference Distance (mm)": ref,
            "Bias (mm)": round(bias, 2),
            "Uncertainty (Std Dev) (mm)": round(std, 2),
            "Mean Measured Distance (mm)": round(mean_dist, 2),
            "95% CI Lower (mm)": round(lower, 2),
            "95% CI Upper (mm)": round(upper, 2)
        })

        # print metrics
        print(f"[CSV] Ref={ref} mm: Bias={bias:.2f} mm, Uncertainty={std:.2f} mm")
        print(f"      Mean={mean_dist:.2f} mm ±{cov:.2f} mm; [{lower:.2f}, {upper:.2f}] mm")

    # prepare data for boxplot
    refs = sorted(errors_by_ref.keys())
    data = [errors_by_ref[r] for r in refs]

    # boxplot
    plt.figure(figsize=(12, 8))
    plt.boxplot(data, positions=refs, widths=4, whis=[5, 95])
    plt.xlabel('Reference Distance (mm)')
    plt.ylabel('Measurement Error (mm)')
    plt.title('Error Distribution vs. Reference Distance')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

    # output metrics as table
    df = pd.DataFrame(metrics).sort_values("Reference Distance (mm)")
    print("\nSummary Table:")
    print(df.to_string(index=False))

    # export table to CSV
    df.to_csv("metrics_summary.csv", index=False)
    print("\nMetrics summary exported to 'metrics_summary.csv'")


if __name__ == '__main__':
    explore_csv_logs()

**Individual Centre Log**

In [ ]:
import pyrealsense2 as rs
import numpy as np
import cv2 as cv
import time
import matplotlib.pyplot as plt
from datetime import datetime

# Configuration Parameters
DEPTH_MIN, DEPTH_MAX = 0.49, 3.5               # Depth range (m)
DEPTH_FOV_H, DEPTH_FOV_V = 70, 55              # Degrees
WIDTH, HEIGHT = 640, 480                       # Frame resolution
FPS = 30
ROI_WIDTH_M, ROI_HEIGHT_M = 0.02, 0.02         # ROI size in meters
SPATIAL_THRESHOLD = 0.20                      # Dynamic depth margin ratio
DURATION_S = 60                                # Total recording duration (seconds)
INTERVAL_S = 1                                 # Recording interval (seconds)

# Log file
LOG_FILENAME = "center_distance_log.txt"

# Initialize RealSense pipeline
pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.depth, WIDTH, HEIGHT, rs.format.z16, FPS)
config.enable_stream(rs.stream.color, WIDTH, HEIGHT, rs.format.bgr8, FPS)
profile = pipeline.start(config)

depth_sensor = profile.get_device().first_depth_sensor()
depth_scale = depth_sensor.get_depth_scale()

# Frame center coordinates
cx, cy = WIDTH // 2, HEIGHT // 2

# State variables
ref_dist = None
records = []
recording = False
start_time = None
next_record_time = None
iteration = 0
max_iterations = int(DURATION_S / INTERVAL_S)

print("Press 's' to start automatic recording, 'f' to set reference during display, 'q' to quit.")
# Create display window
cv.namedWindow("ROI Measurement", cv.WINDOW_AUTOSIZE)

# Main loop
while True:
    current_time = time.time()

    # Fetch frames
    frames = pipeline.wait_for_frames()
    depth_frame = frames.get_depth_frame()
    color_frame = frames.get_color_frame()
    if not depth_frame or not color_frame:
        continue

    # Dynamic ROI based on center distance
    d_center = depth_frame.get_distance(cx, cy)
    if d_center <= 0:
        d_center = DEPTH_MAX
    mpp_x = 2 * d_center * np.tan(np.radians(DEPTH_FOV_H/2)) / WIDTH
    mpp_y = 2 * d_center * np.tan(np.radians(DEPTH_FOV_V/2)) / HEIGHT
    rw = max(1, int(ROI_WIDTH_M / mpp_x))
    rh = max(1, int(ROI_HEIGHT_M / mpp_y))
    x1 = max(0, cx - rw//2)
    y1 = max(0, cy - rh//2)
    x2 = min(WIDTH-1, cx + rw//2)
    y2 = min(HEIGHT-1, cy + rh//2)

    # Extract depth ROI and compute average in mm
    depth_arr = np.asanyarray(depth_frame.get_data(), dtype=np.float32) * depth_scale
    roi = depth_arr[y1:y2+1, x1:x2+1]
    zmin = max(DEPTH_MIN, d_center * (1 - SPATIAL_THRESHOLD))
    zmax = min(DEPTH_MAX, d_center * (1 + SPATIAL_THRESHOLD))
    mask = (roi >= zmin) & (roi <= zmax)
    vals = roi[mask]
    avg_mm = float(np.mean(vals)) * 1000 if vals.size else 0.0

    # Display ROI and overlay
    img = np.asanyarray(color_frame.get_data())
    cv.rectangle(img, (x1, y1), (x2, y2), (0,255,0), 2)
    cv.putText(img, f"Avg: {avg_mm:.1f} mm", (x1, y1-10), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0),2)
    if ref_dist is not None:
        diff = avg_mm - ref_dist
        cv.putText(img, f"Δ {diff:.1f} mm", (x1, y2+20), cv.FONT_HERSHEY_SIMPLEX,0.6,(0,0,255),2)
    cv.imshow("ROI Measurement", img)

    # If recording started, handle automatic logging
    if recording:
        elapsed = current_time - start_time
        if iteration >= max_iterations or elapsed >= DURATION_S:
            break
        if current_time >= next_record_time:
            if ref_dist is None:
                ref_dist = avg_mm
                print(f"Reference set at start: {ref_dist:.2f} mm")
            diff = avg_mm - ref_dist
            ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            records.append((ts, avg_mm, diff))
            with open(LOG_FILENAME, 'a') as f:
                f.write(f"{ts},{avg_mm:.2f},{diff:.2f}\n")
            print(f"[{iteration+1}] {ts}, Avg={avg_mm:.2f} mm, Δ={diff:.2f} mm")
            iteration += 1
            next_record_time += INTERVAL_S

    # Handle key input
    key = cv.waitKey(1) & 0xFF
    if key == ord('s') and not recording:
        recording = True
        start_time = time.time()
        next_record_time = start_time
        iteration = 0
        records.clear()
        print("Started automatic recording...")
    elif key == ord('f'):
        ref_dist = avg_mm
        print(f"Reference manually set: {ref_dist:.2f} mm")
    elif key == ord('q'):
        break

# Cleanup
pipeline.stop()
cv.destroyAllWindows()

# Post-processing if any records
if records:
    inds = list(range(1, len(records)+1))
    dists = [r[1] for r in records]
    diffs = [r[2] for r in records]

    plt.figure()
    plt.plot(inds, diffs, 'o-', label='Error')
    plt.title('Measurement Error Over Time')
    plt.xlabel('Sample #')
    plt.ylabel('Δ (mm)')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    plt.figure()
    plt.plot(inds, dists, 's-', label='Distance')
    plt.title('Measured Distance Over Time')
    plt.xlabel('Sample #')
    plt.ylabel('Distance (mm)')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Compute statistics
    bias = np.mean(diffs)
    std = np.std(diffs, ddof=1)
    mean_dist = ref_dist + bias
    cov = 1.96 * std
    lower = mean_dist - cov
    upper = mean_dist + cov

    print(f"Bias (mean error): {bias:.2f} mm")
    print(f"Uncertainty (std dev): {std:.2f} mm")
    print(f"Mean measured distance: {mean_dist:.2f} mm ± {cov:.2f} mm; [{lower:.2f}, {upper:.2f}] mm")


# Main System

In [ ]:
import pyrealsense2 as rs
import numpy as np
import cv2 as cv
import csv
import pandas as pd
import os
import time
from plyfile import PlyData, PlyElement  # pip install plyfile
import matplotlib.pyplot as plt
import open3d as o3d
import math
from datetime import datetime

'''Configuration Parameters'''
DEPTH_MIN, DEPTH_MAX = 0.49, 3.5                    # Depth range in meters
DEPTH_HORIZONTAL_FOV, DEPTH_VERTICAL_FOV = 70, 55   # Degrees
DEPTH_WIDTH, DEPTH_HEIGHT = 640, 480                # Depth frame resolution
COLOR_WIDTH, COLOR_HEIGHT = 640, 480                # Color frame resolution
FPS_DESIRED = 30                                    # Frames per second

# === ROI shape toggle ===
ROI_SHAPE = "circle"   # "rect" or "circle"

''' Desired ROI dimensions (in meters) '''
a_target_m = 3.090e-2  # radius in meter max 0.098 Area = 301.72 cm^2
roi_width_meters, roi_height_meters = 0.15, 0.15

center_x, center_y = DEPTH_WIDTH // 2, DEPTH_HEIGHT // 2
OUTPUT_DIRECTORY = "captures"

# default inversion mode: 'solve_E' (know P0 → compute E) or 'solve_P0' (know E → compute P0)
JET_INVERSION_MODE = "solve_E"     # or "solve_P0"

KNOWN_P0_PA = 0.9e6                # P0 (jet stagnation pressure) for solve_E: e.g., 0.96 MPa
KNOWN_E_PA  = 40.9e3               # Ei (Young's modulus) for solve_P0: e.g., 12 kPa phantom

# -------- Validation settings --------
VALID_THRESH_FRAC = 0.06   # % band

# ===== Choose mode & known quantity =====
#   "solve_E"  → you know P0, compute E
#   "solve_P0" → you know E, compute P0
# --- Simple stress–strain model constants ---
PHANTOM_THICKNESS_M = 0.020  # h0 [m], set to your phantom thickness
MIN_EPS = 1e-6                # avoid divide-by-zero when strain is tinyKNOWN_P0_PA directly

# --- Jet geometry/spread (for k from Uc and dimp) ---
JET_HALF_ANGLE_DEG = 11.0   # α ≈ 10–12°
USE_FIXED_K = False         # now compute k from Uc & dimp (True = use slide value)
K_SPREAD_S  = 77.6          # only used if USE_FIXED_K = True

# --- Standoff (H) source: 'live' uses center_depth, 'fixed' uses constant ---
# --- Fixed-H calibration (for H_MODE == "fixed") ---
H_MODE = "live"                  # "live" or "fixed"
H_FIXED_NOMINAL = 0.500          # m, nominal 0.5 m
H_FIXED_BIAS_M   = -0.00011      # m, subtract 0.11 mm → 0.49989 m used
H_FIXED_USE = H_FIXED_NOMINAL + H_FIXED_BIAS_M

# Uncertainty band around the *true* reference (500.11 mm ± 1.36 mm)
H_TRUE_REF_M   = 0.50011         # m
H_TRUE_TOL_M   = 0.00136         # m
H_TRUE_MIN_M   = H_TRUE_REF_M - H_TRUE_TOL_M   # 0.49875 m
H_TRUE_MAX_M   = H_TRUE_REF_M + H_TRUE_TOL_M   # 0.50147 m

# Noise reduction & downsampling parameters
SPATIAL_THRESHOLD = 0.10         # Outlier margin
nb_neighbors = 20                # Neighbors for noise reduction
std_ratio = 5.0                  # Std noise reduction factor
voxel_size = 0.0005              # Voxel downsampling size (in meters)

# --- Experiment constants (fixed) ---
# --- Live H (standoff) from LiDAR center depth ---
H_SMOOTH_ALPHA = 0.3     # 0..1, higher = less smoothing (more responsive)
H_MIN, H_MAX    = 0.05, 1.50  # sanity clamp in meters
H_live          = None   # will be initialized on first frame

# Update interval for on-frame information (seconds)
TIME_PRINT_STATEMENT = 5

''' --- LOGGING --- '''
# seconds between automatic logs
TIME_LOGGING = 1/30    
        
# total duration of an auto-logging session
AUTO_LOG_DURATION = 300
     
''' === Auto‑logging & capture state === '''
auto_capture_type = None     # either 'pcd' or 'ply' during a session

# state for timed auto-logging
auto_logging_active = False
auto_log_start_time = None
         
DEBUG_MODE = False

# === Logging & reference state ===
ref_distance = None                   # holds ROI distance when 'f' is pressed
is_logging = False                    # toggles on‑demand logging via 's'
last_auto_log_time = time.time()      # for periodic logging
log_records = []                      # will store tuples (timestamp, ref, delta)
jet_log = []   # tuples: (t, w0, a_used, p_center, E_live, P0_live, mode_str)
force_log = []   # tuples: (t, F_known_from_P0, F_calc_from_solveP0) in Newtons
LOG_ACTIVE_COLOR = (0, 255, 255)      # BGR: cyan/yellowish highlight

# overlay text style
FONT              = cv.FONT_HERSHEY_SIMPLEX
FONT_SCALE_INFO   = 0.5
FONT_SCALE_FPS    = 0.6
THICKNESS_INFO    = 1
THICKNESS_FPS     = 1
PADDING           = 5     # pixels of padding around text
BG_COLOR          = (0, 0, 0)   # black background

# ================== Jet→Pressure→Modulus (bidirectional) ==================
class JetImpingementBidirectional:
    """
    Maps nozzle stagnation pressure P0 → center pressure p_c at the target
    and inverts mechanics (half-space OR thin-plate) to solve either:
      (A) known P0 + measured w0 → estimate E
      (B) known E  + measured w0 → estimate P0
    All derived quantities are recomputed whenever geometry or gas params change.
    """
    def __init__(self,
                 T0_K=298.0, gamma=1.4, R_gas=287.0,
                 rho_amb=1.184,           # ambient ρ at 25C, 1 atm
                 nozzle_d_m=0.002,        # nozzle diameter d
                 H_m=0.50,                # standoff H  (default now 0.50 m)
                 Kdecay=6.0,              # centerline decay constant (~6)
                 a_m=a_target_m,               # loaded radius a
                 nu=0.49,                 # Poisson’s ratio
                 t_m=PHANTOM_THICKNESS_M, # thickness (for thin-plate only)
                 Cbc=1.0/64.0,            # clamped plate coefficient
                 default_mech="halfspace"):
        self.T0, self.gam, self.R = float(T0_K), float(gamma), float(R_gas)
        self.rhoA = float(rho_amb)
        self.d, self.H, self.K = float(nozzle_d_m), float(H_m), float(Kdecay)
        self.a, self.nu, self.t, self.Cbc = float(a_m), float(nu), float(t_m), float(Cbc)
        self.Ue = math.sqrt(self.gam * self.R * (self.T0 * (2.0/(self.gam+1.0))))
        self.default_mech = default_mech.lower()
        self._recompute_maps()

    # ---- internal: recompute all derived constants whenever inputs change ----
    def _recompute_maps(self):
        """
        Theory mapping (no empirical slope):
        - Exit static temperature:  Te = T0 * 2/(γ+1)
        - Sonic exit speed:        Ue = sqrt(γ R Te)
        - Potential core length:   Lc ≈ 6 d
        - Centerline speed:        Uc(H) = Ue           if H <= Lc
                                    = Ue * Lc/H         if H >  Lc
        - Exit density at choke:   ρe = (P0/(R T0)) * (2/(γ+1))**(1/(γ-1))
        - Stagnation at center:    p_center = 0.5 * ρe * Uc^2
                                    = [0.5 * Uc^2 * (2/(γ+1))**(1/(γ-1)) / (R T0)] * P0
        → linear map: p_center = S_p0_to_pc(H) * P0
        """
        # exit temperature and speed (choked)
        self.Te = self.T0 * (2.0/(self.gam+1.0))
        self.Ue = math.sqrt(self.gam * self.R * self.Te)

        # centerline speed at standoff H
        Lc = 6.0 * self.d
        H  = max(self.H, 1e-9)
        self.Uc = self.Ue if H <= Lc else self.Ue * (Lc/H)

        # proportionality for ρe vs P0 (choked)
        self.Crho = (2.0/(self.gam+1.0))**(1.0/(self.gam-1.0)) / (self.R*self.T0)

        # final slope P0→p_center (depends on H through Uc)
        self.S_p0_to_pc = 0.5 * (self.Uc**2) * self.Crho

    # ---- setters (optional, if you want to change geometry on the fly) ----
    def set_standoff(self, H_m):
        self.H = float(H_m)
        self._recompute_maps()

    def set_nozzle_d(self, d_m):
        self.d = float(d_m)
        self._recompute_maps()

    def set_loaded_radius(self, a_m):
        self.a = float(a_m)  # no recompute needed for jet, only mechanics

    # --- jet ↔ pressure ---
    def p_center_from_P0(self, P0_Pa):
        return self.S_p0_to_pc * float(P0_Pa)

    def P0_from_p_center(self, p_center_Pa, eps=1e-12):
        return float(p_center_Pa) / max(self.S_p0_to_pc, eps)

    # --- mechanics (half-space or plate) ---
    def _ensure_model(self, model):
        return (self.default_mech if model is None else model).lower()

    def p_required_for_w0(self, w0_m, E_Pa, a_m=None, model=None, eps=1e-12):
        a = self.a if a_m is None else float(a_m)
        w0 = max(float(w0_m), eps)
        mech = self._ensure_model(model)
        if mech == "halfspace":
            # w0 = [2(1-ν^2)/(πE)] p a  →  p = [πE/(2(1-ν^2))] w0/a
            return (math.pi*E_Pa/(2.0*(1.0-self.nu**2))) * (w0/a)
        elif mech == "plate":
            # w0 = Cbc p a^4 / D,  D = E t^3 / [12(1-ν^2)]  → p = w0 D/(Cbc a^4)
            D = E_Pa*(self.t**3)/(12.0*(1.0-self.nu**2))
            return (w0 * D)/(self.Cbc * (a**4))
        else:
            raise ValueError("model must be 'halfspace' or 'plate'.")

    def E_from_w0_and_p(self, w0_m, p_center_Pa, a_m=None, model=None, eps=1e-12):
        a = self.a if a_m is None else float(a_m)
        w0 = max(float(w0_m), eps)
        mech = self._ensure_model(model)
        if mech == "halfspace":
            # E = [2(1-ν^2)/π] (p a / w0)
            return (2.0*(1.0-self.nu**2)/math.pi) * (p_center_Pa*a / w0)
        elif mech == "plate":
            # E = [12(1-ν^2)/t^3] [p a^4 / (Cbc w0)]
            return (12.0*(1.0-self.nu**2)/(self.t**3)) * (p_center_Pa*(a**4)/(self.Cbc*w0))
        else:
            raise ValueError("model must be 'halfspace' or 'plate'.")

    # --- the two “versions” ---
    def solve_E_from_P0_and_w0(self, P0_Pa, w0_m, a_m=None, model=None):
        p_c = self.p_center_from_P0(P0_Pa)
        E   = self.E_from_w0_and_p(w0_m, p_c, a_m=a_m, model=model)
        return E, p_c

    def solve_P0_from_E_and_w0(self, E_Pa, w0_m, a_m=None, model=None):
        p_req = self.p_required_for_w0(w0_m, E_Pa, a_m=a_m, model=model)
        P0    = self.P0_from_p_center(p_req)
        return P0, p_req

# ===== Configure once (adjust to your rig) =====
BIDIR = JetImpingementBidirectional()

def jet_centerline_uc(H_m, Ue, d_noz):
    """
    Uc(x) ≈ Ue for x <= Lc (potential core), otherwise Ue * Lc/x, Lc≈6d.
    """
    H = max(float(H_m), 1e-9)
    Lc = 6.0 * float(d_noz)
    if H <= Lc:
        return float(Ue)            # still in potential core
    else:
        return float(Ue) * (Lc / H) # decay beyond core

def impingement_diameter(H_m, d_noz, alpha_deg):
    """
    d_imp = d + 2 * H * tan(α)
    """
    H = max(float(H_m), 0.0)
    return float(d_noz) + 2.0 * H * math.tan(math.radians(alpha_deg))

modes = {
    "0": {"name": "Default", "laser_power": 84, "z_min": DEPTH_MIN, "z_max": DEPTH_MAX},
    "1": {"name": "Short Range", "laser_power": 60, "z_min": DEPTH_MIN, "z_max": DEPTH_MAX},
    "2": {"name": "Long Range", "laser_power": 90, "z_min": DEPTH_MIN, "z_max": DEPTH_MAX},
    "3": {"name": "Low Ambient Light", "laser_power": 75, "z_min": DEPTH_MIN, "z_max": DEPTH_MAX},
    "4": {"name": "Max Range", "laser_power": 100, "z_min": DEPTH_MIN, "z_max": None},
        }

def align_and_process_frames(frames):
    """Align depth and color frames and return processed frames."""
    align = rs.align(rs.stream.color)
    aligned_frames = align.process(frames)
    depth_frame = aligned_frames.get_depth_frame()
    color_frame = aligned_frames.get_color_frame()
    return depth_frame, color_frame

def configure_laser_power(depth_sensor, power):
    """Configure the laser power for the depth sensor."""
    if depth_sensor.supports(rs.option.laser_power):
        laser_power_range = depth_sensor.get_option_range(rs.option.laser_power)
        power = max(laser_power_range.min, min(laser_power_range.max, power))
        depth_sensor.set_option(rs.option.laser_power, power)
        print(f"Laser power set to: {power}")
    else:
        print("Laser power adjustment is not supported.")

def configure_roi(depth_frame, roi_width_meters, roi_height_meters, mode):
    """Compute an initial ROI (in pixel coordinates) based on a reference depth."""
    z_min_mode = mode.get("z_min", DEPTH_MIN)
    z_max_mode = mode.get("z_max")
    if z_max_mode is None:
        z_max_mode = depth_frame.get_distance(DEPTH_WIDTH // 2, DEPTH_HEIGHT // 2)
    if z_max_mode <= 0:
        z_max_mode = DEPTH_MAX

    meters_per_pixel_x = 2 * z_max_mode * np.tan(np.radians(DEPTH_HORIZONTAL_FOV / 2)) / DEPTH_WIDTH
    meters_per_pixel_y = 2 * z_max_mode * np.tan(np.radians(DEPTH_VERTICAL_FOV / 2)) / DEPTH_HEIGHT

    roi_width_pixels = int(roi_width_meters / meters_per_pixel_x)
    roi_height_pixels = int(roi_height_meters / meters_per_pixel_y)
    
    x1 = max(0, center_x - roi_width_pixels // 2)
    y1 = max(0, center_y - roi_height_pixels // 2)
    x2 = min(DEPTH_WIDTH - 1, center_x + roi_width_pixels // 2)
    y2 = min(DEPTH_HEIGHT - 1, center_y + roi_height_pixels // 2)

    return (x1, y1, x2, y2, DEPTH_MIN, DEPTH_MAX)

def set_camera_mode(depth_sensor, mode_key, depth_frame):
    """Set the camera mode and return an initial ROI."""
    mode = modes.get(mode_key, modes["0"])
    print(f"Setting camera to {mode['name']} mode...")
    
    if "laser_power" in mode:
        configure_laser_power(depth_sensor, mode["laser_power"])
        
    roi = configure_roi(depth_frame, roi_width_meters, roi_height_meters, mode)
    print(f"Initial ROI configured for {mode['name']} mode: {roi}")
    return roi

def make_xy_mask(mode, x1, y1, x2, y2, cx, cy, r_pix, H, W):
    """
    Returns a boolean mask of shape (H, W) selecting pixels inside the ROI.
    - mode=="rect": select the box [x1:x2, y1:y2]
    - mode=="circle": select pixels whose center is within r_pix of (cx,cy)
    """
    mask = np.zeros((H, W), dtype=bool)
    if mode == "rect":
        mask[max(0,y1):min(H,y2+1), max(0,x1):min(W,x2+1)] = True
    else:
        yy, xx = np.ogrid[:H, :W]
        mask = (xx - cx)**2 + (yy - cy)**2 <= (max(1, r_pix)**2)
    return mask

def save_ply(depth_frame, color_frame, roi, output_dir=OUTPUT_DIRECTORY):
    """
    Save PLY using the current ROI mode (rect or circle).
    `roi` is the rectangular dynamic_roi tuple (x1,y1,x2,y2,zmin,zmax) we already compute
    but if ROI_MODE=="circle" we ignore the corners and build a circle mask with cx,cy,r_pix.
    """
    os.makedirs(str(output_dir), exist_ok=True)
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    filename = os.path.join(str(output_dir), f"roi_capture_{timestamp}.ply")

    # Build point cloud arrays
    points = rs.pointcloud()
    points.map_to(color_frame)
    pc = points.calculate(depth_frame)

    verts = np.asanyarray(pc.get_vertices()).view(np.float32).reshape(DEPTH_HEIGHT, DEPTH_WIDTH, 3)
    color_np = np.asanyarray(color_frame.get_data())

    x1,y1,x2,y2,zmin,zmax = roi
    # compute live m/px to convert a_target_m → r_pix
    center_depth = depth_frame.get_distance((x1+x2)//2, (y1+y2)//2)
    med_z = center_depth if center_depth > 0 else (zmin+zmax)/2.0
    mpp_x = 2*med_z*np.tan(np.radians(DEPTH_HORIZONTAL_FOV/2))/DEPTH_WIDTH
    mpp_y = 2*med_z*np.tan(np.radians(DEPTH_VERTICAL_FOV/2))/DEPTH_HEIGHT

    cx = (x1 + x2)//2
    cy = (y1 + y2)//2
    r_pix = pix_radius_from_meters(a_target_m, mpp_x, mpp_y, mode="areapreserve")

    xy_mask = make_xy_mask(ROI_SHAPE, x1, y1, x2, y2, cx, cy, r_pix, DEPTH_HEIGHT, DEPTH_WIDTH)
    z = verts[:,:,2]
    z_mask = (z >= zmin) & (z <= zmax)
    mask = xy_mask & z_mask

    if not np.any(mask):
        print("[WARN] No valid points in ROI for PLY.")
        return

    pts = verts[mask]
    rgb = cv.cvtColor(color_np, cv.COLOR_BGR2RGB)[mask]  # uint8

    # Pack as PLY (with colors)
    N = pts.shape[0]
    vertex_data = np.empty(N, dtype=[('x','f4'),('y','f4'),('z','f4'),
                                     ('red','u1'),('green','u1'),('blue','u1')])
    vertex_data['x'] = pts[:,0]; vertex_data['y'] = pts[:,1]; vertex_data['z'] = pts[:,2]
    vertex_data['red'] = rgb[:,0]; vertex_data['green'] = rgb[:,1]; vertex_data['blue'] = rgb[:,2]

    PlyData([PlyElement.describe(vertex_data, 'vertex')]).write(filename)
    print(f"File saved: {filename}")

def save_pcd(depth_frame, color_frame, roi, output_dir=OUTPUT_DIRECTORY):
    os.makedirs(str(output_dir), exist_ok=True)
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    filename = os.path.join(str(output_dir), f"roi_capture_{timestamp}.pcd")

    points = rs.pointcloud()
    points.map_to(color_frame)
    pc = points.calculate(depth_frame)

    verts = np.asanyarray(pc.get_vertices()).view(np.float32).reshape(DEPTH_HEIGHT, DEPTH_WIDTH, 3)
    color_np = np.asanyarray(color_frame.get_data())

    x1,y1,x2,y2,zmin,zmax = roi
    center_depth = depth_frame.get_distance((x1+x2)//2, (y1+y2)//2)
    med_z = center_depth if center_depth > 0 else (zmin+zmax)/2.0
    mpp_x = 2*med_z*np.tan(np.radians(DEPTH_HORIZONTAL_FOV/2))/DEPTH_WIDTH
    mpp_y = 2*med_z*np.tan(np.radians(DEPTH_VERTICAL_FOV/2))/DEPTH_HEIGHT

    cx = (x1 + x2)//2
    cy = (y1 + y2)//2
    r_pix = pix_radius_from_meters(a_target_m, mpp_x, mpp_y, mode="areapreserve")

    xy_mask = make_xy_mask(ROI_SHAPE, x1, y1, x2, y2, cx, cy, r_pix, DEPTH_HEIGHT, DEPTH_WIDTH)
    z = verts[:,:,2]
    z_mask = (z >= zmin) & (z <= zmax)
    mask = xy_mask & z_mask

    if not np.any(mask):
        print("[WARN] No valid points in ROI for PCD.")
        return

    pts = verts[mask]
    rgb = cv.cvtColor(color_np, cv.COLOR_BGR2RGB)[mask].astype(np.float32)/255.0

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    pcd.colors = o3d.utility.Vector3dVector(rgb)
    o3d.io.write_point_cloud(filename, pcd)
    print(f"File saved: {filename}")


def preview_roi(color_image, depth_frame, roi_coords, depth_scale):
    """
    Display the ROI overlay on the color image and visualize the corresponding depth ROI.
    """
    x1, y1, x2, y2, z_min, z_max = roi_coords
    image_copy = color_image.copy()
    cv.rectangle(image_copy, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv.imshow("Color Frame with ROI", image_copy)

    depth_data = np.asanyarray(depth_frame.get_data())
    depth_data_m = depth_data.astype(np.float32) * depth_scale
    roi_depth = depth_data_m[y1:y2 + 1, x1:x2 + 1]
    roi_depth_colormap = cv.applyColorMap(cv.convertScaleAbs(roi_depth, alpha=255/DEPTH_MAX), cv.COLORMAP_JET)
    cv.imshow("Depth ROI", roi_depth_colormap)

    valid_depths = roi_depth[(roi_depth >= z_min) & (roi_depth <= z_max)]
    if valid_depths.size > 0:
        print(f"Median depth in ROI: {np.median(valid_depths):.2f} meters")
    else:
        print("No valid depth values in ROI.")
    cv.waitKey(0)
    cv.destroyWindow("Color Frame with ROI")
    cv.destroyWindow("Depth ROI")
    
def pix_radius_from_meters(a_m, mpp_x, mpp_y, mode="inscribed"):
    """
    Convert a physical radius a_m [m] to pixel radius r_pix, given the
    current meters-per-pixel scales along X,Y (mpp_x, mpp_y).
    - 'inscribed'  → conservative: r_pix = a_m / max(mpp_x, mpp_y)
    - 'average'    → r_pix = a_m / ((mpp_x + mpp_y)/2)
    - 'areapreserve' → choose r_pix so that disk area in m² equals π a_m²
                       while pixels are rectangular: r_pix = a_m / sqrt(mpp_x*mpp_y)
    """
    if mode == "inscribed":
        return int(round(a_m / max(mpp_x, mpp_y)))
    elif mode == "average":
        return int(round(a_m / ((mpp_x + mpp_y) * 0.5)))
    elif mode == "areapreserve":
        return int(round(a_m / math.sqrt(mpp_x * mpp_y)))
    else:
        raise ValueError("mode must be 'inscribed', 'average', or 'areapreserve'.")

def meters_from_pix_radius(r_pix, mpp_x, mpp_y, mode="inscribed"):
    """Inverse of above; returns physical radius a_m from pixel radius."""
    if mode == "inscribed":
        return r_pix * max(mpp_x, mpp_y)
    elif mode == "average":
        return r_pix * ((mpp_x + mpp_y) * 0.5)
    elif mode == "areapreserve":
        return r_pix * math.sqrt(mpp_x * mpp_y)
    
def shape_mask_for_roi(h, w, shape="rect", circle=None):
    """
    Returns a boolean mask (h x w) selecting pixels inside the chosen shape.
    - shape='rect': full True mask (i.e., use entire rectangle)
    - shape='circle': needs circle={'cx_rel','cy_rel','r_pix'} relative to ROI top-left.
    """
    if shape == "rect" or circle is None:
        return np.ones((h, w), dtype=bool)
    cx = int(circle["cx_rel"])
    cy = int(circle["cy_rel"])
    r  = int(circle["r_pix"])
    yy, xx = np.ogrid[:h, :w]
    return (xx - cx)**2 + (yy - cy)**2 <= r*r

def oob_delta(meas_m: float, ref_m: float, tol_m: float) -> float:
    """
    Returns 0 if |meas-ref| <= tol, else returns the amount beyond tol.
    """
    if not np.isfinite(meas_m):
        return 0.0
    return max(0.0, abs(meas_m - ref_m) - tol_m)
def within_pct(meas, ref, frac=VALID_THRESH_FRAC):
    """True if |meas - ref| <= frac * |ref| and both are finite/positive."""
    if meas is None or ref is None:
        return False
    if not (np.isfinite(meas) and np.isfinite(ref)):
        return False
    if ref == 0:
        return False
    return abs(meas - ref) <= frac * abs(ref)

def safe_to_csv(df, base_name, outdir=OUTPUT_DIRECTORY, retries=3, delay=0.3):
    """
    Save df to outdir/base_name. If the name is locked (Excel open, etc.),
    fall back to a timestamped file. Retries a few times before giving up.
    Returns the final path, or None if it failed.
    """
    os.makedirs(str(outdir), exist_ok=True)
    base_path = os.path.join(str(outdir), base_name)

    # First try the base name with a couple of retries
    for i in range(retries):
        try:
            df.to_csv(base_path, index=False)
            print(f"[INFO] Saved {base_path}")
            return base_path
        except PermissionError:
            time.sleep(delay)

    # Fall back to a timestamped name if the base is still locked
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    alt_name = f"{os.path.splitext(base_name)[0]}_{stamp}.csv"
    alt_path = os.path.join(str(outdir), alt_name)
    for i in range(retries):
        try:
            df.to_csv(alt_path, index=False)
            print(f"[WARN] '{base_name}' locked. Saved as {alt_path}")
            return alt_path
        except PermissionError:
            time.sleep(delay)

    print(f"[ERROR] Could not save CSV after retries. "
          f"Close any app using '{base_name}' and try again.")
    return None

''' ---------------------- Main Capture Script ----------------------- '''

# Prompt user for capture mode (Dynamic or Statistic/Static capture)
capture_type = input("Select capture type: Dynamic capture (d) or Statistic capture (s): ").strip().lower()
is_dynamic_capture = (capture_type == 'd')
if is_dynamic_capture:
    print("Dynamic capture selected.")
else:
    print("Statistic (static) capture selected.")

# Initialize and start the RealSense pipeline.
pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.depth, DEPTH_WIDTH, DEPTH_HEIGHT, rs.format.z16, FPS_DESIRED)
config.enable_stream(rs.stream.color, COLOR_WIDTH, COLOR_HEIGHT, rs.format.bgr8, FPS_DESIRED)
profile = pipeline.start(config)

device = profile.get_device()
depth_sensor = device.first_depth_sensor()
depth_scale = depth_sensor.get_depth_scale()
print(f"Depth Scale is: {depth_scale} meters per unit")

# Select camera mode
print("Select mode:")
for key, mode in modes.items():
    print(f"{key}: {mode['name']}")
mode_key = input("Enter the mode number: ").strip()
if mode_key not in modes:
    print("Invalid mode. Defaulting to 'Default'.")
    mode_key = "0"

# Obtain initial frames to set up camera mode and initial ROI.
frames = pipeline.wait_for_frames()    
depth_frame, color_frame = align_and_process_frames(frames)
initial_roi = set_camera_mode(depth_sensor, mode_key, depth_frame)
current_mode_text = f"Mode: {modes[mode_key]['name']}"

# Variables for FPS calculation and info update
prev_frame_time = time.time()
last_info_update_time = time.time()
info_lines = []  # This will hold the overlay info text

print("Press 'q' to quit, 'o' to preview ROI,"
      "\n 'c' to Change ROI Shape,"
      "\n '1' PCD, '2' PLY,"
      "\n 'f' set Ref., 's' auto-log,"
      "\n 'Shift + 1(!)' auto-log+PCD, 'Shift + 2(@)' auto-log+PLY,"
      "\n 'e' solve_E (known P0), 'p' solve_P0 (known E)")

try:
    while True:
        frames = pipeline.wait_for_frames()
        depth_frame, color_frame = align_and_process_frames(frames)
        if not depth_frame or not color_frame:
            continue

        # Get the original color frame (rs.frame) for saving
        raw_color_frame = color_frame
          
        # Create a numpy copy for display
        display_color = np.asanyarray(color_frame.get_data()).copy()
        
        if display_color.shape[:2] != (DEPTH_HEIGHT, DEPTH_WIDTH):
            display_color = cv.resize(display_color, (DEPTH_WIDTH, DEPTH_HEIGHT))
        
        # Convert depth frame to numpy array and apply depth scale conversion.
        depth_image = np.asanyarray(depth_frame.get_data())
        depth_image_m = depth_image.astype(np.float32) * depth_scale

        # ----- Determine ROI based on capture mode -----
        if is_dynamic_capture:
            # Dynamic ROI update based on actual depth data.
            x1_init, y1_init, x2_init, y2_init, _, _ = initial_roi
            roi_depth_initial = depth_image_m[y1_init:y2_init + 1, x1_init:x2_init + 1]
            valid_depths = roi_depth_initial[roi_depth_initial > 0]
            median_depth = np.median(valid_depths) if valid_depths.size > 0 else DEPTH_MAX
            delta = SPATIAL_THRESHOLD * median_depth
            dynamic_z_min = max(DEPTH_MIN, median_depth - delta)
            dynamic_z_max = min(DEPTH_MAX, median_depth + delta)
            dynamic_mpp_x = 2 * median_depth * np.tan(np.radians(DEPTH_HORIZONTAL_FOV / 2)) / DEPTH_WIDTH
            dynamic_mpp_y = 2 * median_depth * np.tan(np.radians(DEPTH_VERTICAL_FOV / 2)) / DEPTH_HEIGHT

            roi_width_pixels = int(roi_width_meters / dynamic_mpp_x)
            roi_height_pixels = int(roi_height_meters / dynamic_mpp_y)
            x1_dyn = max(0, center_x - roi_width_pixels // 2)
            y1_dyn = max(0, center_y - roi_height_pixels // 2)
            x2_dyn = min(DEPTH_WIDTH - 1, center_x + roi_width_pixels // 2)
            y2_dyn = min(DEPTH_HEIGHT - 1, center_y + roi_height_pixels // 2)
            dynamic_roi = (x1_dyn, y1_dyn, x2_dyn, y2_dyn, dynamic_z_min, dynamic_z_max)

            # Adaptive adjustment based on object size within the ROI.
            roi_depth_region = depth_image_m[y1_dyn:y2_dyn + 1, x1_dyn:x2_dyn + 1]
            mask = (roi_depth_region >= dynamic_z_min) & (roi_depth_region <= dynamic_z_max)
            if np.any(mask):
                r_indices, c_indices = np.where(mask)
                min_r, max_r = np.min(r_indices), np.max(r_indices)
                min_c, max_c = np.min(c_indices), np.max(c_indices)
                new_x1 = x1_dyn + min_c
                new_y1 = y1_dyn + min_r
                new_x2 = x1_dyn + max_c
                new_y2 = y1_dyn + max_r
                padding_m = 0.01
                padding_px_x = int(padding_m / dynamic_mpp_x)
                padding_px_y = int(padding_m / dynamic_mpp_y)
                padded_x1 = max(0, new_x1 - padding_px_x)
                padded_y1 = max(0, new_y1 - padding_px_y)
                padded_x2 = min(DEPTH_WIDTH - 1, new_x2 + padding_px_x)
                padded_y2 = min(DEPTH_HEIGHT - 1, new_y2 + padding_px_y)
                final_x1 = min(x1_dyn, padded_x1)
                final_y1 = min(y1_dyn, padded_y1)
                final_x2 = max(x2_dyn, padded_x2)
                final_y2 = max(y2_dyn, padded_y2)
                dynamic_roi = (final_x1, final_y1, final_x2, final_y2, dynamic_z_min, dynamic_z_max)
                
        else:
            # Static capture: recompute ROI from the current median depth,
            # but DO NOT apply the adaptive object-size expansion.
            x1_init, y1_init, x2_init, y2_init, _, _ = initial_roi

            roi_depth_initial = depth_image_m[y1_init:y2_init + 1, x1_init:x2_init + 1]
            valid_depths = roi_depth_initial[roi_depth_initial > 0]
            median_depth = np.median(valid_depths) if valid_depths.size > 0 else DEPTH_MAX

            delta = SPATIAL_THRESHOLD * median_depth
            dynamic_z_min = max(DEPTH_MIN, median_depth - delta)
            dynamic_z_max = min(DEPTH_MAX, median_depth + delta)

            dynamic_mpp_x = 2 * median_depth * np.tan(np.radians(DEPTH_HORIZONTAL_FOV / 2)) / DEPTH_WIDTH
            dynamic_mpp_y = 2 * median_depth * np.tan(np.radians(DEPTH_VERTICAL_FOV / 2)) / DEPTH_HEIGHT

            roi_width_pixels  = int(roi_width_meters  / dynamic_mpp_x)
            roi_height_pixels = int(roi_height_meters / dynamic_mpp_y)
            x1_dyn = max(0, center_x - roi_width_pixels // 2)
            y1_dyn = max(0, center_y - roi_height_pixels // 2)
            x2_dyn = min(DEPTH_WIDTH - 1, center_x + roi_width_pixels // 2)
            y2_dyn = min(DEPTH_HEIGHT - 1, center_y + roi_height_pixels // 2)
            dynamic_roi = (x1_dyn, y1_dyn, x2_dyn, y2_dyn, dynamic_z_min, dynamic_z_max)
            
        # --- Live ROI size in meters (use current median depth for m/px) ---
        mpp_x = 2 * median_depth * np.tan(np.radians(DEPTH_HORIZONTAL_FOV / 2)) / DEPTH_WIDTH
        mpp_y = 2 * median_depth * np.tan(np.radians(DEPTH_VERTICAL_FOV / 2)) / DEPTH_HEIGHT
        real_roi_width  = (x2_dyn - x1_dyn) * mpp_x
        real_roi_height = (y2_dyn - y1_dyn) * mpp_y
        
        # ----- Draw ROI rectangle with conditional color and overlay ROI depth -----
        # Compute ROI center and get its depth:
        roi_center_x = (x1_dyn + x2_dyn) // 2
        roi_center_y = (y1_dyn + y2_dyn) // 2
        
        '''Center distance on the frame'''
        center_depth = depth_frame.get_distance(roi_center_x, roi_center_y)
        
        # Is center depth within the characterized uncertainty band?
        H_in_band_now = (
            H_MODE == "fixed"
            and center_depth > 0
            and (H_TRUE_MIN_M <= center_depth <= H_TRUE_MAX_M)
        )
        # Determine color first (kept your logic)
        roi_color_rect = (0, 255, 0) if DEPTH_MIN <= center_depth <= DEPTH_MAX else (0, 0, 255)
        
        # --- draw ROI according to ROI_SHAPE ---
        if ROI_SHAPE == "rect":
            cv.rectangle(display_color, (x1_dyn, y1_dyn), (x2_dyn, y2_dyn), roi_color_rect, 2)
            # Anchor for text (top-right of rectangle)
            anchor_x = x2_dyn + 10
            anchor_y = y1_dyn + 10
            circle_info = None
            A_live = (real_roi_width/mpp_x) * (real_roi_height/mpp_y)  # Pixel^2 (rect)
            area_m2 = max(0.0, real_roi_width) * max(0.0, real_roi_height)
            # conservative: inscribed circle of the rectangle
            a_live = 0.5 * min(real_roi_width, real_roi_height)
        else:
            # circular ROI: center at rect center, area-preserving pixel radius
            cx_pix = (x1_dyn + x2_dyn) // 2
            cy_pix = (y1_dyn + y2_dyn) // 2
            r_pix  = pix_radius_from_meters(a_target_m, mpp_x, mpp_y, mode="areapreserve")
            cv.circle(display_color, (cx_pix, cy_pix), r_pix, roi_color_rect, 2)
            # Text anchor near circle
            anchor_x = cx_pix + r_pix + 10
            anchor_y = cy_pix - r_pix + 10
            # circle params relative to ROI top-left (needed for saving functions)
            circle_info = {
                "cx_rel": cx_pix - x1_dyn,
                "cy_rel": cy_pix - y1_dyn,
                "r_pix":  r_pix
            }
            # --- circular ROI (use inscribed circle) ---
            a_live = a_target_m                                     # radius [m]
            area_m2 = math.pi * a_live**2                            # disk area [m^2]
            A_pixel = math.pi * r_pix**2                            # pixels [m^2]

        # Overlay the distance on the top right of the ROI rectangle
        distance_text = f"{center_depth*1000:.0f} mm"
        line_y = anchor_y
        line_step = 22  # adjust spacing (px) so lines don't overlap
        cv.putText(display_color, distance_text, (anchor_x, line_y),
                cv.FONT_HERSHEY_TRIPLEX, 0.6, roi_color_rect, 1)
        line_y += line_step
        
        # -------- Standoff H (measured vs fixed, with calibration) ----------
        if H_MODE == "live":
            H_meas = center_depth if center_depth > 0 else median_depth
            if not H_meas or math.isinf(H_meas) or math.isnan(H_meas):
                H_meas = H_live if (H_live and H_live > 0) else H_FIXED_USE

            if H_live is None:
                H_live = H_meas
            else:
                H_live = H_SMOOTH_ALPHA * H_meas + (1.0 - H_SMOOTH_ALPHA) * H_live

            H_use = min(max(H_live, H_MIN), H_MAX)

        else:  # H_MODE == "fixed" → apply the 0.11 mm bias and use constant
            H_use  = H_FIXED_USE
            H_live = H_use  # keep coherent for anything that reads H_live later

        # Apply to jet map
        BIDIR.set_standoff(H_use)
        # --- Uncertainty band check (result is 0 inside band) ---
        delta_oob_m = oob_delta(H_use, H_TRUE_REF_M, H_TRUE_TOL_M)
        
        # --- Simple impact-pressure model areas ---
        # Jet footprint (impact) area: A_imp = pi * (d_imp/2)^2,  d_imp = d + 2 H tan(alpha)
        dimp_live = impingement_diameter(H_use, BIDIR.d, JET_HALF_ANGLE_DEG)
        R_imp     = 0.5 * dimp_live
        A_imp     = math.pi * (R_imp ** 2)

        # ROI/load area uses your target radius a_target_m (not the jet radius)
        a_live    = a_target_m
        A_roi     = math.pi * (a_live ** 2)

        # --- Central deflection from LiDAR (m); use median depth relative to reference ---
        E_live_Pa = P0_live_Pa = p_center_Pa = w0 = None
        F_known_live = F_calc_live = None

        if ref_distance is not None:
            # ----- deformation from LiDAR vs reference -----
            if H_in_band_now:
                # inside the 0.5 m uncertainty band → treat Δ and w0 as zero
                delta_mm = 0.0
                w0 = 0.0
            else:
                delta_mm = abs(median_depth - ref_distance) * 1000.0
                w0  = abs(median_depth - ref_distance)

            # strain (used by both modes); MIN_EPS protects other math if needed
            eps = w0 / max(PHANTOM_THICKNESS_M, 1e-9)

            delta_text = f"delta {delta_mm:.0f} mm"
            text_color = LOG_ACTIVE_COLOR if auto_logging_active else roi_color_rect
            cv.putText(display_color, delta_text, (anchor_x, line_y),
                    cv.FONT_HERSHEY_TRIPLEX, 0.6, text_color, 1)
            line_y += line_step
            # --- SIMPLE impact-pressure model ---

            # center pressure from known P0
            pc_known = BIDIR.p_center_from_P0(KNOWN_P0_PA)
            # Force from impact pressure on jet footprint (use A_imp)
            F_known_live = 0.5 * pc_known * area_m2

            # strain from displacement
            eps = w0 / max(PHANTOM_THICKNESS_M, 1e-9)

            if JET_INVERSION_MODE.lower() == "solve_e":
                # Stress on the sample uses ROI area (use A_roi), not the jet footprint
                if w0 == 0.0:
                    E_live_Pa = None
                    ep_main = "E = n/a (w0≈0)"
                else:
                    sigma_known = F_known_live / max(A_roi, 1e-12)
                    E_live_Pa   = sigma_known / max(eps, MIN_EPS)
                    ep_main = f"E = {E_live_Pa/1e3:,.2f} kPa"
                ep_aux = f"P0 = {KNOWN_P0_PA/1e6:.2f} MPa"

            else:  # solve_P0 (known E)
                # Required stress = E * strain (ROI area)
                sigma_req  = KNOWN_E_PA * eps
                F_calc_live = sigma_req * A_roi
                # Back to center impact pressure and then P0
                pc_req     = 2 * F_calc_live / max(A_imp, 1e-12)
                P0_live_Pa = BIDIR.P0_from_p_center(pc_req)

                ep_main = f"P0 = {P0_live_Pa/1e6:.3f} MPa"
                ep_aux  = f"E = {KNOWN_E_PA/1e3:.1f} kPa"

            # overlays (under the Δ line)
            ep_color = LOG_ACTIVE_COLOR if auto_logging_active else (0, 255, 0)
            
            cv.putText(display_color, ep_main, (anchor_x, line_y),
                    cv.FONT_HERSHEY_TRIPLEX, 0.6, ep_color, 1)
            line_y += line_step
            
            cv.putText(display_color, ep_aux, (anchor_x, line_y),
                    cv.FONT_HERSHEY_TRIPLEX, 0.5, ep_color, 1)
            line_y += line_step
            
            if F_known_live is not None:
                cv.putText(display_color, f"F(P0_known) = {F_known_live:.2f} N",
                        (anchor_x, line_y), cv.FONT_HERSHEY_TRIPLEX, 0.5, ep_color, 1)
                line_y += line_step
            
            if F_calc_live is not None:
                cv.putText(display_color, f"F(solve_P0) = {F_calc_live:.2f} N",
                        (anchor_x, line_y), cv.FONT_HERSHEY_TRIPLEX, 0.5, ep_color, 1)
                line_y += line_step
             
            # (Optional) show live strain too
            cv.putText(display_color, f"Strain = {eps:.2f}", (anchor_x, line_y),
                    cv.FONT_HERSHEY_TRIPLEX, 0.5, ep_color, 1)
            r_hint = r_pix if ROI_SHAPE == "circle" else int(0.5*min(x2_dyn-x1_dyn, y2_dyn-y1_dyn))
            if line_y + 5*line_step > DEPTH_HEIGHT:
                line_y = max(20, roi_center_y - r_hint - 10)
                
            # log for post-run plots
            if 'jet_log' not in globals():
                jet_log = []
            
        ''' ----- Overlay Output Information ----- '''
        # Top left: Display current camera mode.
        cv.putText(display_color, current_mode_text, (10, 30),
                   cv.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        jetinv_text = f"JetInv: {'E from P0' if JET_INVERSION_MODE=='solve_E' else 'P0 from E'}"
        cv.putText(display_color, jetinv_text, (10, 55),
           cv.FONT_HERSHEY_SIMPLEX, 0.6, (200, 255, 200), 2)

        if ROI_SHAPE == "rect":
            area = f"ROI Area: {A_live:.4f} Pixel^2"
            area_line = f"(Rect. = {real_roi_width:.2f} x {real_roi_height:.2f} m^2)"
        else:
            area = f"ROI Area: {A_pixel:.3f} Pixel^2"
            area_line = f"(Circle r = {a_live:.3f} m)"
            
        info_lines = [
                    area,
                    f"Median Depth: {median_depth:.3f} m",
                    f"Depth Range: {dynamic_z_min:.3f}-{dynamic_z_max:.3f} m",
                    f"m/px: ({mpp_x:.4f}, {mpp_y:.4f})",
                    area_line
                ]
        
        # Add H-mode and calibration details
        if H_MODE == "fixed":
            # compute out-of-band delta using current center depth
            if center_depth > 0:
                if center_depth < H_TRUE_MIN_M:
                    delta_oob_m = H_TRUE_MIN_M - center_depth
                elif center_depth > H_TRUE_MAX_M:
                    delta_oob_m = center_depth - H_TRUE_MAX_M
                else:
                    delta_oob_m = 0.0
            else:
                delta_oob_m = 0.0

            info_lines += [
                f"H mode: fixed (use = {H_use:.3f} m)",
                          ]
        else:
            info_lines += [f"H mode: live (H = {H_use:.3f} m)"]

        if ref_distance is not None:
            
            current_time = time.time()
        # only-when-close-to-ref samples
        validation_log = []   # list of dicts: {"time":t, "metric":..., "meas":..., "ref":..., "err_pct":...}
        if auto_logging_active and ref_distance is not None:
            elapsed = current_time - auto_log_start_time

            if elapsed <= AUTO_LOG_DURATION:
                if (current_time - last_auto_log_time) >= TIME_LOGGING:
                    # Δ (median vs ref) at this cadence
                    if H_in_band_now:
                        delta = 0.0
                    else:
                        delta = median_depth - ref_distance
                        

                    log_records.append((current_time, ref_distance, delta, w0))

                    # --- Jet inversion at the same cadence ---
                    # --- Same simple model during logging tick ---
                    # Areas at this tick
                    dimp_log   = impingement_diameter(H_use, BIDIR.d, JET_HALF_ANGLE_DEG)
                    R_imp_log  = 0.5 * dimp_log
                    A_imp_log  = math.pi * (R_imp_log ** 2)
                    A_roi_log  = math.pi * (a_target_m ** 2)

                    # Center pressure from known P0 and its force
                    pc_log       = BIDIR.p_center_from_P0(KNOWN_P0_PA)
                    F_known_log  = 0.5 * pc_log * area_m2

                    # strain
                    eps_log = w0 / max(PHANTOM_THICKNESS_M, 1e-9)

                    # E estimate when w0>0 (solve_E view)
                    if w0 == 0.0:
                        E_log = None
                    else:
                        sigma_known_log = F_known_log / max(A_roi_log, 1e-12)
                        E_log = sigma_known_log / max(eps_log, MIN_EPS)

                    # P0 estimate from known E (solve_P0 view)
                    sigma_req_log = KNOWN_E_PA * eps_log
                    F_calc_log    = sigma_req_log * A_roi_log
                    pc_req_log    = 2 * F_calc_log   / max(A_imp_log, 1e-12)
                    P0_log        = BIDIR.P0_from_p_center(pc_req_log)

                    force_log.append((current_time, F_known_log, F_calc_log))
                    jet_log.append((current_time, E_log, P0_log))
                    # ---- Auto-validation (±20% band to reference lines) ----
                    # 1) Elastic modulus vs its ref (if we have E for this tick)
                    if E_log is not None and within_pct(E_log, KNOWN_E_PA):
                        validation_log.append({
                            "time": current_time,
                            "metric": "E",
                            "meas": float(E_log),
                            "ref": float(KNOWN_E_PA),
                            "err_pct": 100.0 * (E_log - KNOWN_E_PA) / KNOWN_E_PA
                        })

                    # 2) Jet supply pressure vs its ref (only in solve_P0 we compute P0_live)
                    if P0_log is not None and within_pct(P0_log, KNOWN_P0_PA):
                        validation_log.append({
                            "time": current_time,
                            "metric": "P0",
                            "meas": float(P0_log),
                            "ref": float(KNOWN_P0_PA),
                            "err_pct": 100.0 * (P0_log - KNOWN_P0_PA) / KNOWN_P0_PA
                        })

                    # 3) Force agreement (need both)
                    if (F_known_log is not None) and (F_calc_log is not None) and within_pct(F_calc_log, F_known_log):
                        validation_log.append({
                            "time": current_time,
                            "metric": "F",
                            "meas": float(F_calc_log),
                            "ref": float(F_known_log),
                            "err_pct": 100.0 * (F_calc_log - F_known_log) / F_known_log
                        })

                    last_auto_log_time = current_time
                    msg = f"[AUTO] t={elapsed:.1f}s  Δ={delta:.3f} m"
                    if E_log is not None:  msg += f" | E={E_log/1e3:.1f} kPa"
                    if P0_log is not None: msg += f" | P0={P0_log/1e6:.3f} MPa"
                    print(msg)

                    # trigger captures if requested
                    if auto_capture_type == 'pcd':
                        print("[AUTO] saving PCD…")
                        save_pcd(depth_frame, raw_color_frame, dynamic_roi, OUTPUT_DIRECTORY)
                    elif auto_capture_type == 'ply':
                        print("[AUTO] saving PLY…")
                        save_ply(depth_frame, raw_color_frame, dynamic_roi, OUTPUT_DIRECTORY)
            else:
                auto_logging_active = False
                auto_capture_type = None
                print(f"[INFO] Completed {AUTO_LOG_DURATION} s auto-logging session.")


        # Render the info lines in the top-right corner.
        start_x = DEPTH_WIDTH - 260
        start_y = 10
        line_height = 22
        
        # compute sizes of each line
        text_sizes = [
            cv.getTextSize(line, FONT, FONT_SCALE_INFO, THICKNESS_INFO)[0]
            for line in info_lines
                    ]
        # maximum line width and total height
        max_w   = max(w for w, h in text_sizes)
        line_h  = text_sizes[0][1] + 5                       # approx. line height
        panel_h = line_h * len(info_lines)

        # panel top-left and bottom-right
        panel_x1 = start_x - PADDING
        panel_y1 = start_y - PADDING
        panel_x2 = start_x + max_w + PADDING
        panel_y2 = start_y + panel_h + PADDING

        # background rectangle
        cv.rectangle(display_color,
                    (panel_x1, panel_y1),
                    (panel_x2, panel_y2),
                    BG_COLOR,
                    thickness=-1)

        # now draw each line on top
        for idx, line in enumerate(info_lines):
            y = start_y + idx * line_h + text_sizes[0][1]
            cv.putText(display_color,
                    line,
                    (start_x, y),
                    FONT,
                    FONT_SCALE_INFO,
                    (255, 255, 255),    # white text
                    THICKNESS_INFO)
            
        # ----- Calculate and Overlay FPS on the Bottom Right -----
        new_frame_time = time.time()
        fps = 1.0 / (new_frame_time - prev_frame_time) if new_frame_time != prev_frame_time else 0
        prev_frame_time = new_frame_time
        # prepare FPS text
        fps_text = f"FPS: {fps:.1f}"
        (fw, fh), baseline = cv.getTextSize(fps_text, FONT, FONT_SCALE_FPS, THICKNESS_FPS)

        # compute bottom‑right position
        fps_x = DEPTH_WIDTH - fw - 10
        fps_y = DEPTH_HEIGHT - 10

        # compute rectangle corners around FPS text
        rect_x1 = fps_x - PADDING
        rect_y1 = fps_y - fh - PADDING
        rect_x2 = fps_x + fw + PADDING
        rect_y2 = fps_y + PADDING

        # draw background
        cv.rectangle(display_color,
                    (rect_x1, rect_y1),
                    (rect_x2, rect_y2),
                    BG_COLOR,
                    thickness=-1)

        # draw FPS text on top
        cv.putText(display_color,
                fps_text,
                (fps_x, fps_y),
                FONT,
                FONT_SCALE_FPS,
                (0, 204, 51),        # original green
                THICKNESS_FPS)

        # ----- Combine Color and Depth Displays for Preview -----
        combined_depth = cv.applyColorMap(cv.convertScaleAbs(depth_image_m, alpha=255/DEPTH_MAX), cv.COLORMAP_JET)
        combined_image = np.hstack((cv.resize(display_color, (640, 480)), combined_depth))
        cv.imshow('RealSense Stream', combined_image)

        # Optionally update console info every TIME_PRINT_STATEMENT seconds (if needed)
        if time.time() - last_info_update_time > TIME_PRINT_STATEMENT:
            last_info_update_time = time.time()

        key = cv.waitKey(1) & 0xFF
        
        # — set reference distance —
        if key == ord('f'):
            ref_distance = median_depth
            last_auto_log_time = time.time()   # restart the 5 s counter
            print(f"[INFO] Reference set to {ref_distance:.3f} m")
            
        # — start/stop on‑demand logging —
        elif key == ord('s'):
            if ref_distance is None:
                print("[WARN] Set reference first (press 'f').")
            else:
                auto_logging_active = True
                auto_log_start_time = time.time()
                last_auto_log_time = auto_log_start_time
                auto_capture_type    = None
                print(f"[INFO] Auto-logging Δ every {TIME_LOGGING}s for {AUTO_LOG_DURATION}s.")
        # — Δ + PCD auto‑logging (‘!’) —
        elif key == ord('!'):  # Shift+1
            if ref_distance is None:
                print("[WARN] Set reference first (press 'f').")
            else:
                auto_logging_active = True
                auto_log_start_time = time.time()
                last_auto_log_time = auto_log_start_time
                auto_capture_type = 'pcd'
                print(f"[INFO] Auto‑logging+PCD every {TIME_LOGGING}s for {AUTO_LOG_DURATION}s.")
                
        # — Δ + PLY auto‑logging (‘@’) —
        elif key == ord('@'):  # Shift+2
            if ref_distance is None:
                print("[WARN] Set reference first (press 'f').")
            else:
                auto_logging_active = True
                auto_log_start_time = time.time()
                last_auto_log_time = auto_log_start_time
                auto_capture_type = 'ply'
                print(f"[INFO] Auto‑logging+PLY every {TIME_LOGGING}s for {AUTO_LOG_DURATION}s.")
                
        if key == ord('q'):
            break
        elif key == ord('o'):
            print("Previewing dynamic ROI...")
            preview_roi(display_color, depth_frame, dynamic_roi, depth_scale)
        elif key == ord('c'):
            ROI_SHAPE = "rect" if ROI_SHAPE == "circle" else "circle"
            print(f"[INFO] ROI shape set to {ROI_SHAPE}")
        elif key == ord('1'):
            print("Saving point cloud as PCD (using grayscale intensity)...")
            save_pcd(depth_frame, raw_color_frame, dynamic_roi, OUTPUT_DIRECTORY)
        elif key == ord('2'):
            print("Saving point cloud as custom PLY (with RGB and intensity)...")
            save_ply(depth_frame, raw_color_frame, dynamic_roi, OUTPUT_DIRECTORY)
        elif key == ord('e'):
            JET_INVERSION_MODE = "solve_E"
            if ref_distance is None:
                print("[MODE] solve_E selected. Set reference first (press 'f') to compute E.")
            else:
                print("[MODE] solve_E selected. Computing E from known P0.")
        elif key == ord('p'):
            JET_INVERSION_MODE = "solve_P0"
            if ref_distance is None:
                print("[MODE] solve_P0 selected. Set reference first (press 'f') to compute P0.")
            else:
                print("[MODE] solve_P0 selected. Computing P0 from known E.")
        elif key == ord('h'):
            H_MODE = "fixed" if H_MODE == "live" else "live"
            print(f"[INFO] H mode set to '{H_MODE}' "
                f"({'%.3f m (biased)'%H_FIXED_USE if H_MODE=='fixed' else 'center depth'})")

finally:
    pipeline.stop()
    cv.destroyAllWindows()
    print("Pipeline stopped.")

# Convert logs into DataFrames
df_log   = pd.DataFrame(log_records, columns=["time", "ref_m", "delta_m", "abs_delta_m"])
df_jet   = pd.DataFrame(jet_log,    columns=["time", "E_Pa", "P0_Pa"])
df_force = pd.DataFrame(force_log,  columns=["time", "F_known_N", "F_live_N"])

# Normalize elapsed time
t0 = min(df_log["time"].min(), df_jet["time"].min(), df_force["time"].min())
df_log["elapsed_time_s"]   = df_log["time"] - t0
df_jet["elapsed_time_s"]   = df_jet["time"] - t0
df_force["elapsed_time_s"] = df_force["time"] - t0

# Merge (outer join so you don’t lose values)
df_all = df_log.merge(df_jet, on="elapsed_time_s", how="outer").merge(df_force, on="elapsed_time_s", how="outer")

# Drop raw "time" columns if not needed
df_all = df_all.drop(columns=["time_x","time_y","time"])

# --- Save combined log into timestamped CSV ---
def _find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

if df_all is not None and not df_all.empty:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    full_filename = f"calibration_data_{timestamp}.csv"
    df_all.to_csv(full_filename, index=False)
    print(f"[INFO] Full log saved → {full_filename}")

    # --- Ensure we have delta_mm ---
    # Try to derive from meters if needed
    if 'delta_mm' not in df_all.columns:
        m_col = _find_col(df_all, ['delta_m', 'delta', 'w0_m'])
        if m_col:
            df_all['delta_mm'] = df_all[m_col] * 1000.0
        else:
            # Nothing to derive from → create NaNs so the filter will drop all
            df_all['delta_mm'] = np.nan
            print("[WARN] Could not find delta field to make delta_mm.")

    # --- Ensure we have ref_mm (reference distance in mm) ---
    ref_mm_col = _find_col(df_all, ['ref_mm'])
    if ref_mm_col is None:
        ref_m_col = _find_col(df_all, ['ref_m', 'reference_m', 'ref_distance_m'])
        ref_mm_col = 'ref_mm'
        if ref_m_col:
            df_all['ref_mm'] = df_all[ref_m_col] * 1000.0
        else:
            # Some logs stored ref as a constant; try 'ref' or 'reference'
            ref_unitless_col = _find_col(df_all, ['ref', 'reference'])
            if ref_unitless_col:
                # assume meters
                df_all['ref_mm'] = df_all[ref_unitless_col] * 1000.0
            else:
                # Fall back to a single known ref (if you used ref_distance variable)
                try:
                    df_all['ref_mm'] = float(ref_distance) * 1000.0
                except Exception:
                    df_all['ref_mm'] = np.nan
                    print("[WARN] Could not find/derive reference distance; validation may be empty.")
                    
    EPS_MM     = 1e-3   # treat near-zero as zero
    # --- Build validation mask: 0 < |Δ| ≤ 20% * ref ---
    mask_valid = (
        df_all['delta_mm'].abs() > EPS_MM
    ) & (
        df_all['delta_mm'].abs() <= VALID_THRESH_FRAC * df_all['ref_mm'].abs()
    )

    df_valid = df_all.loc[mask_valid].copy()

    # Keep the useful columns if present (will ignore missing ones)
    wanted_cols = [
        # time
        'elapsed_s', 'time_s', 't', 'timestamp',
        # geometry / depth
        'center_depth_m', 'median_depth_m', 'ref_m', 'ref_mm', 'delta_mm',
        # mechanics
        'E_Pa', 'E_kPa',
        'P0_Pa', 'P0_MPa',
        # forces
        'F_known_N', 'F_live_N', 'F_calc_N', 'F_N',
        # any other context you store
        'mode', 'H_use_m'
    ]
    df_valid = df_valid[[c for c in wanted_cols if c in df_valid.columns]]

    valid_filename = f"validation_delta_only_{int(VALID_THRESH_FRAC*100)}pct_{timestamp}.csv"
    if not df_valid.empty:
        df_valid.to_csv(valid_filename, index=False)
        print(f"[INFO] Validation (Δ-only, ≤{int(VALID_THRESH_FRAC*100)}% ref & Δ≠0) saved → {valid_filename}")
        print(f"[INFO] Kept {len(df_valid)} / {len(df_all)} samples.")
    else:
        print("[INFO] No samples matched the Δ-only validation criteria.")
else:
    print("[INFO] Nothing to save: df_all is empty.")

# After pipeline.stop()
if log_records:
    times, refs, deltas, abs_delta = zip(*log_records)
    t0 = times[0]
    elapsed = [t - t0 for t in times]

    distances_m = [r + d for r, d in zip(refs, deltas)]

    refs_mm      = [d * 1000 for d in refs]
    deltas_mm    = [d * 1000 for d in deltas]
    abs_deltas_mm = [d * 1000 for d in abs_delta]
    distances_mm = [d * 1000 for d in distances_m]
    
    plt.figure(figsize=(12,8))
    #plt.subplot(2,1,1)
    plt.plot(elapsed, distances_mm, marker='x', linestyle='--', label='Distance (mm)')
    plt.plot(elapsed, refs_mm,      marker='o', linestyle='-',  label='Reference (mm)')
    plt.ylabel('Distance (mm)')
    plt.title('Distance & Reference vs Time')
    plt.legend(); plt.grid(True)
    
    plt.figure(figsize=(12,8))
    #plt.subplot(2,1,2)
    plt.plot(elapsed, abs_deltas_mm, marker='o', linestyle='-')
    plt.xlabel('Elapsed time (s)')
    plt.ylabel('Δ distance (mm)')
    plt.title(f'Δ Displacement vs Time with ROI {A_roi:.4f} m^2')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
    # === Post-run jet inversion plots (cadence = TIME_LOGGING) ===
    if 'jet_log' in globals() and len(jet_log) > 0:
        t_arr, E_arr, P0_arr = zip(*jet_log)
        t0 = t_arr[0]
        jelapsed = [t - t0 for t in t_arr]

        E_kPa  = np.array([ (e/1e3) if (e is not None) else np.nan for e in E_arr ])
        P0_MPa = np.array([ (p/1e6) if (p is not None) else np.nan for p in P0_arr ])
        
        def highlight_ref(ax, y, label, color, band=None):
            """
            Draw a bold ref line with optional translucent band.
            band: absolute +/- value (same units as y) or None
            """
            if band and band > 0:
                ax.axhspan(y-band, y+band, color=color, alpha=0.12, linewidth=0, zorder=0,
                        label=f"{label} ± {band:g}")
            # “glow” effect: wide faint line underneath + crisp line on top
            ax.axhline(y, color=color, linewidth=6, alpha=0.25, zorder=3)
            ax.axhline(y, color=color, linewidth=2.5, linestyle="--", zorder=4, label=label)

        # ----- E plot -----
        fig, ax = plt.subplots(figsize=(12,8))
        if np.isfinite(E_kPa).any():
            ax.plot(jelapsed, E_kPa, marker='o', linestyle='-', label='E (kPa)', zorder=2)

        E_ref_kPa = KNOWN_E_PA/1e3
        highlight_ref(ax, E_ref_kPa, "E ref", color="#ff7f0e", band=0)

        ax.set_xlim(0, max(jelapsed) if len(jelapsed)>0 else 1)
        ax.set_xlabel("Elapsed time (s)"); ax.set_ylabel("E (kPa)")
        ax.set_title("Elastic Modulus vs Time"); ax.grid(True, zorder=1)
        ax.legend(); plt.tight_layout(); plt.show()

        # ----- P0 plot -----
        fig, ax = plt.subplots(figsize=(12,8))
        if np.isfinite(P0_MPa).any():
            ax.plot(jelapsed, P0_MPa, marker='s', linestyle='-', label='P0 (MPa)', zorder=2)

        P0_ref_MPa = KNOWN_P0_PA/1e6
        highlight_ref(ax, P0_ref_MPa, "P0 ref", color="#9467bd", band=0)

        ax.set_xlim(0, max(jelapsed) if len(jelapsed)>0 else 1)
        ax.set_xlabel("Elapsed time (s)"); ax.set_ylabel("P0 (MPa)")
        ax.set_title("Jet Supply Pressure vs Time"); ax.grid(True, zorder=1)
        ax.legend(); plt.tight_layout(); plt.show()
        
        # === Correlation between F(P0_known) and F(solve_P0) ===
        # ----- Force vs Time (F_known vs F_live) -----
        if 'force_log' in globals() and len(force_log) > 0:
            tf, Fk, Fc = zip(*force_log)
            t0f = tf[0]
            felapsed = [t - t0f for t in tf]

            Fk_arr = np.array(Fk, dtype=float)  # F_known from P0_known
            Fc_arr = np.array(Fc, dtype=float)  # F_live  from E*eps (solve_P0 chain)

            plt.figure(figsize=(12,8))
            # plot only finite points to avoid gaps if NaNs exist
            mk = np.isfinite(Fk_arr)
            mc = np.isfinite(Fc_arr)

            if mk.any():
                plt.plot(np.array(felapsed)[mk], Fk_arr[mk],
                        marker='o', linestyle='-', label='F_known (pc from P0)')
            if mc.any():
                plt.plot(np.array(felapsed)[mc], Fc_arr[mc],
                        marker='s', linestyle='-', label='F_live (from E·ε)')

            plt.xlabel('Elapsed time (s)')
            plt.ylabel('Force (N)')
            plt.title('Jet Force vs Time')
            plt.grid(True)
            plt.legend()
            plt.tight_layout()
            plt.show()
            
    else:
        print("[INFO] No jet-inversion records to plot.")
else:
    print("No log records to plot.")

# RGB & Grayscale Display

In [ ]:
import os
import open3d as o3d
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
import matplotlib.colors as mcolors  # For converting RGB to hex strings
import webbrowser

POINTS = 10000
OUTPUT_DIRECTORY = "captures"
HTML_OUTPUT = "point_cloud_visualization.html"

# Define a common grayscale colorscale.
grayscale = [[0, "rgb(0,0,0)"], [1, "rgb(255,255,255)"]]

def load_files_from_folder(folder, extensions=[".ply", ".pcd", ".obj"]):
    """Load all supported files from the specified folder."""
    files = [os.path.join(folder, f) for f in os.listdir(folder)
             if any(f.lower().endswith(ext) for ext in extensions)]
    if not files:
        print("No supported files found in the folder.")
        return []
    files.sort()  # Sort files alphabetically for consistent ordering
    return files

def align_point_cloud(pcd, origin=(0, 0, 0)):
    """Apply a rotation matrix to align the point cloud and set a new origin."""
    rotation_matrix = o3d.geometry.get_rotation_matrix_from_xyz((np.pi / 2, 1.85 * np.pi, np.pi))
    pcd.rotate(rotation_matrix, center=(0, 0, 0))
    translation_vector = np.array(origin) - np.mean(np.asarray(pcd.points), axis=0)
    pcd.translate(translation_vector)
    return pcd

def load_file_as_numpy(file_path, origin=(0, 0, 0)):
    """
    Load a point cloud file (PLY, PCD, or OBJ) and convert it to numpy arrays.
    Returns:
      points: Nx3 numpy array of point coordinates.
      colors: Nx3 numpy array of colors (if available) or None.
    """
    try:
        if file_path.lower().endswith((".ply", ".pcd")):
            pcd = o3d.io.read_point_cloud(file_path)
        elif file_path.lower().endswith(".obj"):
            mesh = o3d.io.read_triangle_mesh(file_path)
            pcd = mesh.sample_points_uniformly(number_of_points=POINTS)
        else:
            print(f"Unsupported file format: {file_path}")
            return np.array([]), None

        if len(pcd.points) == 0:
            print(f"Warning: {file_path} is empty.")
            return np.array([]), None

        pcd = align_point_cloud(pcd, origin)
        points = np.asarray(pcd.points)
        colors = np.asarray(pcd.colors) if pcd.has_colors() and len(pcd.colors) > 0 else None
        return points, colors
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return np.array([]), None

def compute_intensity(points):
    """Fallback function to compute intensity from point distances."""
    distances = np.linalg.norm(points, axis=1)
    if distances.max() - distances.min() > 0:
        norm_intensity = (distances - distances.min()) / (distances.max() - distances.min())
    else:
        norm_intensity = np.zeros_like(distances)
    return norm_intensity

def create_scatter(points, colors, use_rgb=False):
    """
    Create a Plotly Scatter3d object.
    
    If use_rgb is False, marker color is set using grayscale intensity.
    If True, marker color is set using the actual RGB data (converted to hex strings).
    """
    x, y, z = points[:, 0], points[:, 1], points[:, 2]
    if not use_rgb:
        if colors is not None and colors.size > 0:
            intensity = colors[:, 0]
        else:
            intensity = compute_intensity(points)
        marker_dict = dict(
            size=2,
            color=intensity,
            colorscale=grayscale,
            opacity=0.8,
            cmin=0,
            cmax=1,
            showscale=True,
            colorbar=dict(
                title="Intensity",
                titleside="right",
                tickvals=[0, 0.5, 1],
            )
        )
    else:
        if colors is not None and colors.size > 0:
            rgb_list = [mcolors.to_hex(color) for color in colors]
        else:
            intensity = compute_intensity(points)
            rgb_list = [mcolors.to_hex([i, i, i]) for i in intensity]
        marker_dict = dict(
            size=2,
            color=rgb_list,
            opacity=0.8,
            showscale=False
        )
    return go.Scatter3d(
        x=x,
        y=y,
        z=z,
        mode="markers",
        marker=marker_dict,
        text=[f"x: {xi:.2f}, y: {yi:.2f}, z: {zi:.2f}" for xi, yi, zi in zip(x, y, z)],
        hoverinfo="text"
    )

def create_figure(points, colors, file_label, color_mode="Grayscale", scale=1.5):
    """Create a Plotly figure given point cloud data and a selected color mode.
    
    The function computes the centroid of the point cloud and the maximum distance
    from the centroid. It then sets uniform axis ranges (with a margin defined by 
    the scale factor) and computes tick spacing accordingly. The camera eye position 
    is set relative to the centroid and the object's size.
    """
    use_rgb = (color_mode == "RGB")
    scatter = create_scatter(points, colors, use_rgb=use_rgb)
    
    # Compute the centroid and maximum distance (i.e. extent) from the centroid.
    center = points.mean(axis=0)
    max_distance = np.max(np.linalg.norm(points - center, axis=1))
    
    # Define axis ranges uniformly based on the centroid and the scale factor.
    x_range = [center[0] - scale * max_distance, center[0] + scale * max_distance]
    # Note: y and z axes are swapped relative to our transformation.
    y_range = [center[2] - scale * max_distance, center[2] + scale * max_distance]
    z_range = [center[1] - scale * max_distance, center[1] + scale * max_distance]
    
    # Compute tick spacing based on these ranges.
    dtick_x = (x_range[1] - x_range[0]) / 10 if (x_range[1]-x_range[0]) != 0 else 0.02
    dtick_y = (y_range[1] - y_range[0]) / 10 if (y_range[1]-y_range[0]) != 0 else 0.02
    dtick_z = (z_range[1] - z_range[0]) / 10 if (z_range[1]-z_range[0]) != 0 else 0.02
    
    # Compute a relative camera position:
    # Position the camera diagonally away from the center by 2 times the maximum distance.
    camera_eye = dict(
        x=1,
        y=1,
        z=1
    )
    
    fig = go.Figure(data=[scatter])
    fig.update_layout(
        title=f"File: {file_label} ({color_mode})",
        scene=dict(
            xaxis_title="X (meters)",
            yaxis_title="Z (meters)",  # (Due to rotation, our original Y becomes Z)
            zaxis_title="Y (meters)",
            xaxis=dict(range=x_range, dtick=dtick_x),
            yaxis=dict(range=y_range, dtick=dtick_y),
            zaxis=dict(range=z_range, dtick=dtick_z),
            camera=dict(eye=camera_eye)
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )
    return fig

''' --------------------------------------------------
# Main: Build Interactive UI Using Radio Buttons
 -------------------------------------------------- '''

all_files = load_files_from_folder(OUTPUT_DIRECTORY)
all_data = []  # List of tuples: (file_path, points, colors)
if not all_files:
    print("No files found for visualization.")
else:
    for f in all_files:
        pts, cols = load_file_as_numpy(f)
        if pts.size > 0:
            all_data.append((f, pts, cols))
    if not all_data:
        print("No valid point cloud data to display.")

if all_data:
    # Create a list of file labels for the radio buttons.
    file_labels = [os.path.basename(item[0]) for item in all_data]
    
    # Radio buttons for file selection.
    file_radio = widgets.RadioButtons(
        options=file_labels,
        description="Select File:",
        value=file_labels[0]
    )
    
    # Radio buttons for color mode selection.
    color_radio = widgets.RadioButtons(
        options=["Grayscale", "RGB"],
        description="Color Mode:",
        value="Grayscale"
    )
    
    # Button to save the current view as HTML.
    save_html_button = widgets.Button(
        description="Save as HTML",
        button_style='info'
    )
    
    output_area = widgets.Output()
    global current_fig
    current_fig = None

    def update_plot(*args):
        global current_fig
        with output_area:
            output_area.clear_output(wait=True)
            selected_label = file_radio.value
            color_mode = color_radio.value
            # Find the matching data for the selected file.
            for (file_path, pts, cols) in all_data:
                if os.path.basename(file_path) == selected_label:
                    fig = create_figure(pts, cols, selected_label, color_mode=color_mode)
                    fig.show()
                    current_fig = fig
                    break

    def save_html_callback(b):
        global current_fig
        if current_fig is None:
            print("No figure to save.")
            return
        html_str = current_fig.to_html(full_html=True, include_plotlyjs='cdn', div_id="graphDiv")
        # Inject a custom "Save Image" button.
        custom_button = """
        <div style="position: absolute; top: 10px; left: 10px; z-index: 9999;">
            <button onclick="Plotly.downloadImage(document.getElementById('graphDiv'), {format: 'png', filename: 'point_cloud_image'});">
                Save Image
            </button>
        </div>
        """
        html_str = html_str.replace("</body>", custom_button + "</body>")
        html_filename = f"point_cloud_{file_radio.value.replace(' ', '_')}.html"
        with open(html_filename, "w") as f:
            f.write(html_str)
        webbrowser.open("file://" + os.path.abspath(html_filename))
        print(f"Saved HTML to {html_filename}")

    # Attach callbacks.
    file_radio.observe(update_plot, names="value")
    color_radio.observe(update_plot, names="value")
    save_html_button.on_click(save_html_callback)

    # Display the controls and the output area.
    ui_box = widgets.HBox([file_radio, color_radio, save_html_button])
    display(ui_box, output_area)
    update_plot()

# Folder containing point cloud files.
folder_path = OUTPUT_DIRECTORY

# PLY VDO

In [ ]:
import os
import time
import numpy as np
import open3d as o3d
import cv2

# suppress warnings (only show errors)
o3d.utility.set_verbosity_level(o3d.utility.VerbosityLevel.Error)

# ---------------- Configuration ----------------
ply_dir        = "PLY_VDO"                    # Folder containing PLYs
video_out_path = "output/ply_playback.mp4"    # Video output path
W, H           = 640, 480                     # Window & nominal video/HUD resolution
INIT_FPS       = 15                           # Initial playback & saving framerate
POINT_SIZE     = 2.0                          # Rendered point size (pixels)
HUD_FONT       = cv2.FONT_HERSHEY_SIMPLEX

# ---------------- Prepare PLY list ----------------
ply_files = sorted(
    os.path.join(ply_dir, f)
    for f in os.listdir(ply_dir)
    if f.lower().endswith(".ply")
)
if not ply_files:
    raise RuntimeError(f"No .ply files found in '{ply_dir}'")
num_frames = len(ply_files)

# ---------------- Playback & Saving State ----------------
current_idx   = 0
playing       = True
saving        = False
should_exit   = False
video_writer  = None
fps           = INIT_FPS

# ---------------- Create Visualizer (interactive window) ----------------
vis = o3d.visualization.VisualizerWithKeyCallback()
vis.create_window("PLY Sequence (Open3D)", width=W, height=H, visible=True)
ctr = vis.get_view_control()

# Render options (point size, bg color)
opt = vis.get_render_option()
opt.point_size = POINT_SIZE
# opt.background_color = np.array([0, 0, 0])  # uncomment for black bg

# Load and add the first point cloud once
base_pcd = o3d.io.read_point_cloud(ply_files[0])
vis.add_geometry(base_pcd)  # camera initialized here

# Initialize camera: “up” is +Y, viewing along −Z, look at model center
bbox   = base_pcd.get_axis_aligned_bounding_box()
center = bbox.get_center()
ctr.set_lookat(center)
ctr.set_front([0.0, 0.0, -1.0])
ctr.set_up([0.0, -1.0, 0.0])   # CHANGED: keep right-side-up (+Y is up)

# ---------------- OpenCV HUD window (NEW) ----------------
cv2.namedWindow("Playback + HUD (OpenCV)", cv2.WINDOW_NORMAL)  # NEW
cv2.resizeWindow("Playback + HUD (OpenCV)", W, H)              # NEW
# Optional: place the HUD window near the Open3D window
# cv2.moveWindow("Playback + HUD (OpenCV)", 60, 60)

# ---------------- Helpers ----------------
def ensure_writer():
    """(Re)create the VideoWriter if needed."""
    global video_writer
    if video_writer is None:
        os.makedirs(os.path.dirname(video_out_path), exist_ok=True)
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        video_writer = cv2.VideoWriter(video_out_path, fourcc, fps, (W, H))

def grab_open3d_rgb():
    """Capture current Open3D window as RGB uint8 image of size (H, W, 3)."""
    img_o3d = vis.capture_screen_float_buffer(do_render=False)  # RGB float [0,1]
    img = (255 * np.asarray(img_o3d)).astype(np.uint8)          # RGB uint8
    # If the visualizer produced a different size (DPI scaling), resize to (W,H)
    if img.shape[1] != W or img.shape[0] != H:
        img = cv2.resize(img, (W, H), interpolation=cv2.INTER_AREA)
    return img

def overlay_hud(rgb_img, fps_now, idx, playing, saving):
    """Draw a high-contrast HUD (FPS, frame idx, state) on rgb_img."""
    hud = rgb_img.copy()
    lines = [
        f"Frame {idx+1}/{num_frames}",
        f"FPS {fps_now:.1f}",
        f"{'PLAY' if playing else 'PAUSE'}  |  {'REC' if saving else 'LIVE'}"
    ]

    # --- Box metrics ---
    font = HUD_FONT
    scale = 0.7
    thick = 2
    pad  = 8
    gap  = 6
    # measure text block
    sizes = [cv2.getTextSize(t, font, scale, thick)[0] for t in lines]
    box_w = max(w for (w, h) in sizes) + 2*pad
    box_h = sum(h for (w, h) in sizes) + (len(lines)-1)*gap + 2*pad

    # top-left corner
    x0, y0 = 10, 10
    # draw semi-transparent black rectangle
    overlay = hud.copy()
    cv2.rectangle(overlay, (x0, y0), (x0+box_w, y0+box_h), (0,0,0), -1)
    alpha = 0.5
    cv2.addWeighted(overlay, alpha, hud, 1-alpha, 0, hud)

    # draw each line
    y = y0 + pad + sizes[0][1]
    for i, text in enumerate(lines):
        cv2.putText(hud, text, (x0+pad, y), font, scale, (255,255,255), thick, cv2.LINE_AA)
        if i < len(lines)-1:
            y += sizes[i+1][1] + gap
    return hud

def clamp(v, lo, hi):
    return max(lo, min(hi, v))

def update_geometry_for_index(idx):
    """Replace base_pcd data in place for the given index (preserves camera)."""
    new_pcd = o3d.io.read_point_cloud(ply_files[idx])
    base_pcd.points = new_pcd.points
    if new_pcd.has_colors():
        base_pcd.colors = new_pcd.colors
    vis.update_geometry(base_pcd)

# ---------------- Key Callbacks (Open3D window) ----------------
def toggle_play(vis_):
    """Space bar: play/pause"""
    global playing
    playing = not playing
    print("[INFO] Playing" if playing else "[INFO] Paused")
    return False

def toggle_save(vis_):
    """S or s: start/stop saving video"""
    global saving, video_writer
    saving = not saving
    if saving:
        ensure_writer()
        print(f"[INFO] Started saving to '{video_out_path}' @ {fps} FPS")
    else:
        if video_writer:
            video_writer.release()
            video_writer = None
        print("[INFO] Stopped saving video.")
    return False

def request_exit(vis_):
    """Q or q: exit"""
    global should_exit
    should_exit = True
    print("[INFO] Exiting playback.")
    return False

def prev_frame(vis_):
    """Left arrow: go to previous frame (when paused)"""
    global current_idx
    if not playing:
        current_idx = (current_idx - 1) % num_frames
        update_geometry_for_index(current_idx)
    return False

def next_frame(vis_):
    """Right arrow: go to next frame (when paused)"""
    global current_idx
    if not playing:
        current_idx = (current_idx + 1) % num_frames
        update_geometry_for_index(current_idx)
    return False

def slower(vis_):
    """'-' key: decrease playback fps"""
    global fps, video_writer
    fps = clamp(fps - 1, 1, 120)
    # Recreate writer with new fps if currently saving
    if saving and video_writer:
        video_writer.release()
        video_writer = None
        ensure_writer()
    print(f"[INFO] FPS → {fps}")
    return False

def faster(vis_):
    """'+' key: increase playback fps"""
    global fps, video_writer
    fps = clamp(fps + 1, 1, 120)
    if saving and video_writer:
        video_writer.release()
        video_writer = None
        ensure_writer()
    print(f"[INFO] FPS → {fps}")
    return False

# Register callbacks
vis.register_key_callback(ord(" "), toggle_play)  # Space
vis.register_key_callback(ord("S"), toggle_save)
vis.register_key_callback(ord("s"), toggle_save)
vis.register_key_callback(ord("Q"), request_exit)
vis.register_key_callback(ord("q"), request_exit)
vis.register_key_callback(262, next_frame)  # GLFW_KEY_RIGHT
vis.register_key_callback(263, prev_frame)  # GLFW_KEY_LEFT
vis.register_key_callback(ord("-"), slower)
vis.register_key_callback(ord("+"), faster)
vis.register_key_callback(ord("="), faster)  # allow '=' (shifted '+')

# ---------------- Main Loop ----------------
try:
    prev_t = time.time()
    while True:
        if should_exit:
            break

        if playing:
            current_idx = (current_idx + 1) % num_frames
            update_geometry_for_index(current_idx)

        # Render the Open3D interactive window
        vis.poll_events()
        vis.update_renderer()

        # Measure instantaneous FPS
        now_t = time.time()
        inst_fps = 1.0 / max(1e-6, (now_t - prev_t))
        prev_t = now_t

        # Grab Open3D frame → overlay HUD → show in OpenCV mirror window
        rgb = grab_open3d_rgb()
        hud = overlay_hud(rgb, inst_fps, current_idx, playing, saving)
        cv2.imshow("Playback + HUD (OpenCV)", cv2.cvtColor(hud, cv2.COLOR_RGB2BGR))

        # Optionally save the same HUD frame to video
        if saving:
            ensure_writer()
            video_writer.write(cv2.cvtColor(hud, cv2.COLOR_RGB2BGR))

        # Keep OpenCV window responsive (press ESC here also quits)
        if cv2.waitKey(1) == 27:  # ESC
            should_exit = True

        # Control playback speed
        time.sleep(1.0 / fps)

finally:
    if video_writer:
        video_writer.release()
    cv2.destroyAllWindows()
    vis.destroy_window()
    print("[INFO] Shut down complete.")